# Repository Structure Walkthrough — Predictive Log Anomaly Engine V3

This notebook is an interactive guide to the repository layout and how modules connect at runtime.

Goal:
- help engineers navigate the codebase quickly
- show where core behavior lives
- map folder structure to execution flow

## 2. Show Repo Tree

The following cell prints a shallow tree for quick orientation.

In [1]:
import os

MAX_DEPTH = 2
EXCLUDE = {'.git', '.venv', '__pycache__', '.pytest_cache'}

def depth(path):
    rel = os.path.relpath(path, '.')
    return 0 if rel == '.' else rel.count(os.sep) + 1

for root, dirs, files in os.walk('.', topdown=True):
    dirs[:] = [d for d in dirs if d not in EXCLUDE]
    d = depth(root)
    if d > MAX_DEPTH:
        dirs[:] = []
        continue
    indent = '  ' * (d - 1)
    print(f"{indent}{root}")

.


## 3. Walkthrough by Section

### Core application
- `src/api/`: FastAPI app factory, routes, settings, and pipeline container wiring.
- `src/runtime/`: inference engines, rolling-window logic, and v2 runtime pipeline.
- `src/modeling/`: model implementations (baseline/transformer/v2 components).
- `src/alerts/`: alert policy, dedup/cooldown, n8n outbox/webhook client.
- `src/observability/`: Prometheus metrics registry and middleware.
- `src/semantic/`: optional semantic enrichment/explanation layer (V3).

### Ops and deployment
- `docker/`: image + compose stack (`docker-compose.yml`, `Dockerfile`).
- `prometheus/`: scrape and alert-rule config.
- `grafana/`: datasource and dashboard provisioning.

### Data/model assets
- `models/`: trained model files loaded at runtime.
- `artifacts/`: thresholds, vocab/template assets, n8n outbox.
- `data/`: raw/intermediate/processed datasets.

### Validation and scripts
- `tests/`: unit/integration/system tests.
- `scripts/`: run-api and data-pipeline helper scripts.

### Optional/historical
- `notebooks/`, `demo/`, `static-demo/`, `examples/`: companion assets.
- `archive/`: historical scripts/assets retained for traceability.

In [2]:
# Show selected top-level folders and immediate children
from pathlib import Path

targets = [
    Path('src'), Path('docker'), Path('prometheus'), Path('grafana'),
    Path('scripts'), Path('tests'), Path('models'), Path('artifacts'), Path('data'),
]

for t in targets:
    if not t.exists():
        continue
    print(f"\n{t}/")
    for child in sorted(t.iterdir()):
        suffix = '/' if child.is_dir() else ''
        print(f"  - {child.name}{suffix}")

## 4. Code Navigation Examples

These imports represent key connection points across API and runtime layers.

In [3]:
import sys
# Navigation imports (may require local dependency compatibility)
project_root = str(Path(__file__).resolve().parent.parent) if '__file__' in dir() else str(Path('.').resolve().parent)
if project_root not in sys.path:
	sys.path.append(project_root)

create_app = None
InferenceEngine = None

try:
	from src.api.app import create_app
	print('create_app ->', create_app)
except Exception as e:
	print('create_app import failed:', repr(e))

try:
	from src.runtime.inference_engine import InferenceEngine
	print('InferenceEngine ->', InferenceEngine)
except Exception as e:
	print('InferenceEngine import failed:', repr(e))
	if "partially initialized module 'torch'" in str(e):
		print("Detected torch circular-init issue. Restart the kernel and run this import cell again.")

create_app -> <function create_app at 0x000001B0C44B23E0>
InferenceEngine -> <class 'src.runtime.inference_engine.InferenceEngine'>


Connection logic:
- `create_app()` (in `src/api/app.py`) wires settings, middleware, and routers.
- `Pipeline` (in `src/api/pipeline.py`) is attached to app state and used by routes.
- `InferenceEngine` (in `src/runtime/inference_engine.py`) owns windowing/scoring/threshold decisions.
- `AlertManager` and `N8nWebhookClient` handle alert policy and dispatch side effects.

## 5. Flow Mapping

Logical execution flow across folders:

`main.py` -> `scripts/run_api.py` -> `src/api/app.py`
-> `src/api/routes.py` / `src/api/pipeline.py`
-> `src/runtime/*` + `src/modeling/*`
-> `src/alerts/*`
-> `src/observability/metrics.py` and infra (`prometheus/`, `grafana/`)

Versioned extensions:
- `/v2/*` route family uses `src/runtime/pipeline_v2.py` and `src/runtime/inference_engine_v2.py`.
- `/v3/*` route family exposes semantic/status APIs and enrichment outputs from `src/semantic/*`.

In [4]:
# Optional: inspect key files quickly from inside the notebook
from pathlib import Path

key_files = [
    Path('main.py'),
    Path('scripts/run_api.py'),
    Path('src/api/app.py'),
    Path('src/api/pipeline.py'),
    Path('src/runtime/inference_engine.py'),
]

for p in key_files:
    print(p, 'exists=' + str(p.exists()))

main.py exists=False
scripts\run_api.py exists=False
src\api\app.py exists=False
src\api\pipeline.py exists=False
src\runtime\inference_engine.py exists=False


## 6. Summary

Suggested onboarding path for developers:
1. Start with `main.py` and `scripts/run_api.py` to understand service startup.
2. Read `src/api/app.py` and `src/api/routes.py` for request lifecycle.
3. Read `src/api/pipeline.py` for component orchestration.
4. Read `src/runtime/inference_engine.py` (and optionally `src/runtime/pipeline_v2.py`) for internals.
5. Read `src/alerts/manager.py` and `src/observability/metrics.py` for operational behavior.

This structure keeps runtime paths explicit while preserving optional/demo/historical assets in separate areas.